In [1]:
import jax.numpy as jnp

In [2]:
def relu(x):
    return jnp.maximum(0,x)
def softmax(x):
    return jnp.exp(x)/jnp.sum(jnp.exp(x),axis=0)

# L=−[y⋅log(y^)+(1−y)⋅log(1−y^)]

def cross_entropy(x,y):
    return -jnp.sum(y*jnp.log(x))

In [6]:
x=jnp.arange(6.0)
print(relu(x))
print(softmax(x))
print(cross_entropy(softmax(x),x))

[0. 1. 2. 3. 4. 5.]
[0.00426978 0.01160646 0.03154963 0.08576079 0.233122   0.6336913 ]
26.8429


In [9]:
from jax import random
"""
传统 Python random 模块：依赖全局状态，每次调用 random() 会修改全局状态，无法精确复现结果；
JAX random 模块:随机数完全由「密钥(Key)」决定 —— 相同的 Key 会生成完全相同的随机序列，且不会修改原始 Key
"""
key=random.PRNGKey(124)
print(key)
key,subkey = random.split(key)
print(key)
print(subkey)

[  0 124]
[ 101608377 3691553362]
[ 677605301 2363350609]


In [12]:
# 功能：生成符合标准正态分布（均值为 0，标准差为 1）的随机数数组(包含 100 万个元素的一维数组)
x = random.normal(key,(1_000_000,))
# IPython/Jupyter 环境中的魔术命令.多次运行后面的代码，并计算平均执行时间
%timeit relu(x)

108 μs ± 3.11 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [13]:
from jax import jit

jitted_relu = jit(relu)
a = jitted_relu(x)  #complies on first call
%timeit relu(a)         #未即时编译
# %timeit jitted_relu(x)  #即时编译

104 μs ± 533 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [20]:
# 自定义一个激活函数进行性能测试
def selu(x,alpha=1.67,lmbda=1.05):
    return lmbda * jnp.where(x>0,x,alpha*jnp.exp(x)-alpha)

key = random.key(1000)                  # (等价于random.PRNGKey) 生成根密钥（seed=1000，确保可复现）
x = random.normal(key,(1_000_000,))     # 生成 100 万个正态分布随机数
%timeit selu(x).block_until_ready()     # 未即时编译   无论函数是否经过 JIT 编译，JAX 对数组的操作默认都是 “先构建计算图，不立即计算”，直到需要获取结果时才执行实际运算。


726 μs ± 11.8 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [21]:

selu_jit = jit(selu)
_=selu_jit(x)                           #complies on first call    JIT 第一次执行会包含编译时间，建议先预热一次
%timeit selu_jit(x).block_until_ready() #即时编译


175 μs ± 1.18 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [ ]:
# matrix multiplication(矩阵乘法测试)   未jit
def mm(x,y):
    return jnp.dot(x,y)

# matrix multiplication(矩阵乘法测试)   jit编译
@jit
def mm_jit(x,y):
    return jnp.dot(x,y)


In [26]:
key = random.key(1000)  
a = random.normal(key,(1000,1000))  # 1000x1000 正态分布矩阵  ab相同
b = random.normal(key,(1000,1000))  # 1000x1000 正态分布矩阵  ab相同

%timeit mm(a,b).block_until_ready()     #未即时编译


1.76 ms ± 26.8 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [27]:
_ = mm_jit(a, b).block_until_ready()    # jit先预热进行缓存：执行编译+计算，
%timeit mm_jit(a,b).block_until_ready() #即时编译

1.77 ms ± 51.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
